# Data Cleaning and Initial Exploration

## 1. Read Data Files



In [78]:
import pandas as pd
import numpy as np
import plotly.express as px
Own_Funds_Original=pd.read_csv("SQ_Own_Funds.csv",encoding="latin1")
Balance_Sheet_Original=pd.read_csv("SQ_Balance_Sheet.csv",encoding="latin1")
Premiums_Original=pd.read_csv("SQ_Premiums_Claims_Expenses.csv",encoding="latin1")

In [79]:
Own_Funds_Original.head()

,Reporting country,Reference period,Item code,Item name,Value,Undertaking type,Date of extraction (yyyymmdd),"Number of submissions (per reporting country, reference date and undertaking type)"
0,AUSTRIA,2016 Q3,R0500,Total available own funds to meet the SCR,856.112328,Life undertakings,20250930,7
1,AUSTRIA,2016 Q3,R0500,Total available own funds to meet the SCR,16353.071138,Non-Life undertakings,20250930,15
2,AUSTRIA,2016 Q3,R0500,Total available own funds to meet the SCR,15943.253781,Other undertakings,20250930,19
3,AUSTRIA,2016 Q3,R0510,Total available own funds to meet the MCR,856.112328,Life undertakings,20250930,7
4,AUSTRIA,2016 Q3,R0510,Total available own funds to meet the MCR,16316.286682,Non-Life undertakings,20250930,15


### Create Dictionary of Codes

In [80]:
Balance_Sheet_Original['Item code'] = Balance_Sheet_Original['Item code'].astype(str) + '_BS'
Premiums_Original['Item code'] = Premiums_Original['Item code'].astype(str) + '_P'
Own_Funds_Original['Item code'] = Own_Funds_Original['Item code'].astype(str) + '_O'

In [81]:
Codes = (
    pd.concat([
        Balance_Sheet_Original[['Item code', 'Item name']],
        Premiums_Original[['Item code', 'Item']].rename(columns={'Item': 'Item name'}),
        Own_Funds_Original[['Item code', 'Item name']]
    ])
    .drop_duplicates(subset=['Item code'])
    .reset_index(drop=True))

In [82]:
Codes.tail(2)

,Item code,Item name
155,R0540 - C0040_O,EOF Tier 2
156,R0540 - C0050_O,EOF Tier 3


####Fix the Reference Period

In [83]:
Balance_Sheet_Original['Year'] = Balance_Sheet_Original['Reference period'].str.extract(r'(\d{4})').astype(int)
Balance_Sheet_Original['Quarter'] = Balance_Sheet_Original['Reference period'].str.extract(r'(Q\d)')[0]
Balance_Sheet_Original['Date'] = pd.PeriodIndex(    Balance_Sheet_Original['Year'].astype(str) + Balance_Sheet_Original['Quarter'],freq='Q').to_timestamp()
Balance_Sheet_Original.drop(columns=['Year', 'Quarter'], inplace=True)

In [ ]:
Premiums_Original['Year'] = Premiums_Original['Reference period'].str.extract(r'(\d{4})').astype(int)
Premiums_Original['Quarter'] = Premiums_Original['Reference period'].str.extract(r'(Q\d)')[0]
Premiums_Original['Date'] = pd.PeriodIndex(Premiums_Original['Year'].astype(str) + Premiums_Original['Quarter'],freq='Q').to_timestamp()
Premiums_Original.drop(columns=['Year', 'Quarter'], inplace=True)

In [ ]:
Own_Funds_Original['Year'] = Own_Funds_Original['Reference period'].str.extract(r'(\d{4})').astype(int)
Own_Funds_Original['Quarter'] = Own_Funds_Original['Reference period'].str.extract(r'(Q\d)')[0]
Own_Funds_Original['Date'] = pd.PeriodIndex(Own_Funds_Original['Year'].astype(str) + Own_Funds_Original['Quarter'],freq='Q').to_timestamp()
Own_Funds_Original.drop(columns=['Year', 'Quarter'], inplace=True)

In [ ]:
Own_Funds_Original

## 2. Restructure the data bases

1. Use only the columns we are interested

In [ ]:
Balance_Sheet = Balance_Sheet_Original[['Reporting country','Date', 'Value','Item code']]
Premiums = Premiums_Original[['Reporting country','Date', 'Value','Item code']]
Own_Funds = Own_Funds_Original[['Reporting country','Date', 'Value','Item code']]



2.   Long to Wide format



In [ ]:
Balance_Sheet_wide = (
    Balance_Sheet.pivot_table(
        index=['Reporting country', 'Date'],
        columns='Item code',
        values='Value',
        aggfunc='sum',           # ✅ sum duplicates
        fill_value=0             # optional: replace NaNs with 0
    )
    .reset_index()
)

Premiums_wide = (
    Premiums.pivot_table(
        index=['Reporting country', 'Date'],
        columns='Item code',
        values='Value',
        aggfunc='sum',           # ✅ sum duplicates
        fill_value=0
    )
    .reset_index()
)

Own_Funds_wide = (
    Own_Funds.pivot_table(
        index=['Reporting country', 'Date'],
        columns='Item code',
        values='Value',
        aggfunc='sum',           # ✅ sum duplicates
        fill_value=0
    )
    .reset_index()
)


In [ ]:
Balance_Sheet_wide.head(2)

In [ ]:
Premiums_wide.head(2)

3.   Exclude 'EEA' from the 'Countries' in Premiums



In [ ]:
Premiums_wide=Premiums_wide[Premiums_wide['Reporting country']!='EEA']
Own_Funds_wide=Own_Funds_wide[Own_Funds_wide['Reporting country']!='EEA']

## 3. Merge the Data Bases

In [ ]:
df = pd.merge(Balance_Sheet_wide, Premiums_wide, on=["Reporting country", "Date"], how="outer")
df = pd.merge(df, Own_Funds_wide, on=["Reporting country", "Date"], how="outer")
df.head(3)

In [ ]:
print("Balance_Sheet shape:", Balance_Sheet_wide.shape)
print("Premiums shape:", Premiums_wide.shape)
print("Own_Funds shape:", Own_Funds_wide.shape)
print("df shape:", df.shape)

## 4. Missing Values and format

1. Replace N/A with 0


In [ ]:
df = df.fillna(0)

2.   Make sure all columns except 'Reporting country' and 'Date' are float and round to 2 decimals.



In [ ]:
df.info()

In [ ]:
df

In [ ]:
num_cols = df.columns.difference(['Reporting country', 'Date'])
df[num_cols] = df[num_cols].astype(float)
df[num_cols] = df[num_cols].round(2)

In [ ]:
num_cols = df.columns.difference(['Reporting country', 'Date'])

for col in num_cols:
    # mask of values that can't be converted
    mask = pd.to_numeric(df[col], errors='coerce').isna() & df[col].notna()
    if mask.any():
        print(f"Column: {col}")
        print(df.loc[mask, col])


In [ ]:
df.tail(4)

## 5. Check for duplicates

In [ ]:
duplicates = df[df.duplicated(subset=['Reporting country', 'Date'], keep=False)]
print(duplicates)
print(f"\nTotal duplicated rows: {len(duplicates)}")


# Research question 3: data processing

In [ ]:
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt

columns_of_interest = [
    "Reporting country",
    "Date",
    "R0130_BS", #investments in bonds
    "R0100_BS", #investments in equities
    "R0500_BS", #total assets
    "R0200_P", # written premiums for non-life
    "R1500_P", # written premiums for life
    "R0080_BS", #Property (other than for own use)
    "R0090_BS", #Holdings in related undertakings, including participations
    "R0180_BS", #Collective Investments Undertakings
    "R0190_BS", #Derivatives 
    "R0200_BS", # Deposits other than cash equivalents 
    "R0210_BS" #Other investments 
]
data = df[columns_of_interest].copy()

# Total premiums written is the sum of written premiums for non-life and written premiums for life
data["Total premiums written"] = data["R0200_P"] + data["R1500_P"]
#Composition of the assets depending on the type of investments:
#Bonds 
data["Bonds"] = data["R0130_BS"]
#Equities
data["Equities"] = data["R0100_BS"]
#Property (other than for own use)
data["Property"] = data["R0080_BS"]
#Holding in related undertakings, including participation
data["Holding related undertakings"] = data["R0090_BS"]
data["Collective investments undertakings"] = data["R0180_BS"]
data["Derivatives"] = data["R0190_BS"]
#Deposits other than cash equivalents
data["Deposits"] = data["R0200_BS"]
data["Other investments"] = data["R0210_BS"]

## Figure 1

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Parameters
main_countries = ['FRANCE', 'GERMANY', 'ITALY']
investment_vars = [
    'Bonds','Equities','Property','Holding related undertakings',
    'Collective investments undertakings','Derivatives','Deposits','Other investments'
]
default_investment = 'Bonds'
n_quantiles = 3
# Preprocessing
data['Date'] = pd.to_datetime(data['Date'])

data_main  = data[data['Reporting country'].isin(main_countries)].copy()
data_other = data[~data['Reporting country'].isin(main_countries)].copy()

# Country clusters based on total premiums
country_prem = (
    data_other.groupby('Reporting country')['Total premiums written']
    .sum()
    .reset_index()
)

country_prem['cluster'] = pd.qcut(
    country_prem['Total premiums written'],
    q=n_quantiles,
    labels=['Low premiums','Medium premiums','High premiums']
)

country_to_cluster = dict(zip(country_prem['Reporting country'], country_prem['cluster']))
data_other['Premium Cluster'] = data_other['Reporting country'].map(country_to_cluster)

# France, Italy and Germany get their own cluster
data_main['Premium Cluster'] = 'Main countries (FR, IT, DE)'

data_all = pd.concat([data_main, data_other], ignore_index=True)

# Filter DataFrames by cluster
df_low   = data_all[data_all['Premium Cluster'] == 'Low premiums']
df_med   = data_all[data_all['Premium Cluster'] == 'Medium premiums']
df_high  = data_all[data_all['Premium Cluster'] == 'High premiums']
df_main2 = data_all[data_all['Premium Cluster'] == 'Main countries (FR, IT, DE)']


# Plot
def create_figure(df, title):
    fig = go.Figure()
    countries = sorted(df['Reporting country'].unique())

    # Add one trace per country
    for c in countries:
        temp = df[df['Reporting country'] == c].sort_values('Date')
        fig.add_trace(go.Scatter(
            x=temp['Date'],
            y=temp[default_investment],
            mode='markers',
            marker=dict(size=6),
            name=c
        ))

    # Dropdown to switch investment variable
    buttons = []
    for inv in investment_vars:
        y_new = [df[df['Reporting country']==c].sort_values('Date')[inv].tolist()
                 for c in countries]

        buttons.append(dict(
            label=inv,
            method='update',
            args=[{"y": y_new}, {"yaxis": {"title": inv}}]
        ))

    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title=default_investment,
        updatemenus=[dict(buttons=buttons, x=1.05, y=1.15)],
        height=500,
        width=800   
    )
    return fig


# Create and show figures
create_figure(df_low,   "Low Premium Countries").show()
create_figure(df_med,   "Medium Premium Countries").show()
create_figure(df_high,  "High Premium Countries").show()
create_figure(df_main2, "Main Countries: France, Italy, Germany").show()


## Figure 1.2?? Table containing information of cluster for Volume Premiums per Country, in order to classify different EU countries

In [ ]:
# Total premiums for countries (except France, Italy, Germany)
country_prem = (
    data_other
    .groupby('Reporting country', observed=True)['Total premiums written']
    .sum()
    .reset_index()
)

# Recreate clusters
country_prem['Cluster'] = pd.qcut(
    country_prem['Total premiums written'],
    q=3,
    labels=['Low premiums', 'Medium premiums', 'High premiums']
)

# Summary table for clusters
cluster_table = (
    country_prem
    .groupby('Cluster', observed=True)['Total premiums written']
    .agg(
        min='min',
        max='max',
        mean='mean',
        count='count'
    )
    .reset_index()
)
# replace "count" by "number of countries"
cluster_table = cluster_table.rename(columns={'count': 'Number of countries'})

# Add 3 countries left in a row
main_totals = (
    data_main
    .groupby('Reporting country')['Total premiums written']
    .sum()
)

main_row = pd.DataFrame([{
    'Cluster': 'Main countries (FR, IT, DE)',
    'min': main_totals.min(),
    'max': main_totals.max(),
    'mean': main_totals.mean(),
    'Number of countries': len(main_totals)
}])

# Combine cluster table + main countries
cluster_table_final = pd.concat([cluster_table, main_row], ignore_index=True)

for col in ['min', 'max', 'mean']:
    cluster_table_final[col] = cluster_table_final[col].map(lambda x: f"{x:,.0f}")

# show final table
cluster_table_final


## Figure 2: Correlation matrix between assets types, total premiums written, per year

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

data['Date'] = pd.to_datetime(data['Date'])
data['Year'] = data['Date'].dt.year

# Columns to include in correlation
num_cols = [
    'Total premiums written', 'Bonds', 'Equities', 'Property',
    'Holding related undertakings', 'Collective investments undertakings',
    'Derivatives', 'Deposits', 'Other investments'
]

# Keep only existing columns
num_cols = [c for c in num_cols if c in data.columns]

years = sorted(data['Year'].unique())

# Compute correlation for each year
corr_matrices = {}
text_matrices = {}

for year in years:
    df_y = data[data['Year'] == year][num_cols]
    corr = df_y.corr().round(2)
    corr_matrices[year] = corr.values
    text_matrices[year] = corr.astype(str).values


# Create figure (default = latest year)
default_year = years[-1]

fig = go.Figure()

fig.add_trace(go.Heatmap(
    z=corr_matrices[default_year],
    x=num_cols,
    y=num_cols,
    colorscale="RdBu_r",        # blue = -1, red = +1
    zmin=-1,
    zmax=1,
    text=text_matrices[default_year],
    texttemplate="%{text}",   # show values
    textfont={"size":12, "color":"black"},
    colorbar=dict(title="Correlation")
))

# Dropdown to select years
buttons = []

for year in years:
    buttons.append(
        dict(
            label=str(year),
            method="update",
            args=[
                {
                    "z": [corr_matrices[year]],
                    "text": [text_matrices[year]]
                },
                {
                    "title": f"Correlation Matrix — {year}"
                }
            ]
        )
    )

fig.update_layout(
    title=f"Correlation Matrix — {default_year}",
    xaxis=dict(title="Variables", tickangle=45),
    yaxis=dict(title="Variables", autorange="reversed"),
    width=750,
    height=750,
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1.15, y=1.15
    )]
)

fig.show()
